# 03: Curate & Extract Training Data

This notebook has two stages:

- **Stage 1: Trajectory selection (implemented below).** Join a run's trajectories + evaluations, apply a configurable quality profile, and write the passing subset to `output/curated/run=<id>/profile=<name>/`. This is the shared `selection` package: the `usersim select` CLI runs the exact same code.
- **Stages 2+: Record-shaping (still deferred).** Turning the curated subset into SFT / pairwise / verifier / PRM / online-RL records (`output/training/*.jsonl`) is a separate workstream; the five-type taxonomy contract below documents the planned shapes. `output/curated/` is the *upstream* of that extraction, not a replacement.

## Why record-shaping is still deferred

Per the project plan (and `docs/roadmap.md`), training-record extraction is intentionally a separate workstream that benefits from:

- A stable trajectory schema (now we have it: `simulation_outcome`, `trajectory_id`, `persona_uuid`, `probe_family` / `probe_variant` / `prompt_version`).
- A stable evaluator schema (now we have it: per-axis × per-judge wide cells, probe-specific scorer outputs).
- Stakeholder alignment with the *training* org on the five-type taxonomy below: what gets extracted, what shape the training records take, who owns the QA / governance loop.

Without those alignments, anything we ship here would be retrofitted six weeks from now. So instead we publish the **interface contract** and stop.

## Five-type training-data taxonomy

When this notebook ships, it will extract one or more of:

| Type | Source | Filter | Format |
|------|--------|--------|--------|
| **SFT** (supervised fine-tuning) | High-scoring assistant turns (incl. the `financial_services` **dynamic** tier: grounded, judge-scored answers) | `per_axis_overall_mean >= threshold` per axis × clean simulator status | `(prompt, response)` pairs |
| **Pairwise** (preference pairs) | Matched-pair runs against two assistant models (or two prompt variants) | Different scores on the same `persona_uuid + probe_variant + prompt_version` | `(prompt, chosen, rejected)` |
| **Verifier** | Trajectories with deterministic ground truth: tool-calling **and** the `financial_services` **verifiable** tier (`gold_tool_sequence` + `expected_state_deltas`) | `evaluator/scorers/tool_use` 8-axis scores; finance verifier pass/fail | `(state, action, verifier_label)` |
| **PRM** (process reward) | Per-turn judge outputs from `simulation_traces` | `assistant_quality` trace entries with sufficient n | `(prefix, step, step_score)` |
| **Online RL** | Branching counterfactuals (replay a turn under a different sampling) | Disagreement between branch outcomes | `(state, action_a, action_b, branch_outcomes)` |

## What the extractor will need from upstream notebooks

All of these are already produced by `01_simulate` + `02_evaluate_simulation`:

- `output/trajectories/` (Hive-partitioned by `locale=X/probe_family=Y/`): `simulation_outcome` (status, failure attribution, counters, per-model resource), `simulation_traces` (per-turn / per-call telemetry), `conversation_messages` (the trajectory itself), `persona_uuid`, `trajectory_id`.
- `output/evaluations/` (same Hive partitioning): wide `assistant_eval` cells (per-axis × per-judge scores + scorer outputs).

## What this notebook writes

Now (Stage 1, implemented below):

- `output/curated/run=<id>/profile=<name>/locale=X/probe_family=Y/part-*.parquet`: the passing trajectories (full schema + `selection_*` quality columns)
- `output/curated/run=<id>/profile=<name>/_selection_manifest.json`: the funnel + profile + provenance

Eventually (Stages 2+, record-shaping):

- `output/training/sft.jsonl`: SFT examples
- `output/training/pairwise.jsonl`: preference pairs
- `output/training/verifier.jsonl`: verifier examples
- `output/training/prm.jsonl`: process-reward records
- `output/training/online_rl.jsonl`: branching pairs

Each with provenance (`trajectory_id`, `persona_uuid`, `code_sha`, model identities, `prompt_version`) so a downstream training run can audit the population and pin the inputs.

## Open questions blocking implementation

Tracked in `docs/roadmap.md`:

1. Threshold-tuning protocol for SFT acceptance (per-axis, per-probe, per-locale).
2. Pairwise sourcing: matched-pair vs counterfactual-branch, and the cost of either.
3. Whose review process gates the verifier / PRM corpora before training picks them up.
4. Governance for `assets/safety_corpus/` (the agentic-safety probe will produce sensitive verifier records).
5. Goodhart's-law mitigation: what slice of the population is held out from training-data extraction so we can measure model improvement honestly?

Until those are settled, this notebook is a contract, not code.

In [ ]:
# Stage 1: Trajectory selection (load + select; preview & write come next).
#
# Drives the shared `selection` package (identical to `usersim select`):
#   load_runs -> select_trajectories -> [preview] -> write_curated_dataset -> [push_to_hf]
#
# Tune the profile knobs below; the GENEROUS default throws out only clear
# violations/errors. Re-run with a different profile name and the output is
# namespaced (`profile=<name>/`) so prior curations are never clobbered.

import dataclasses

from selection import load_runs, select_trajectories
from selection.profiles import available_profiles, get_profile

print("available profiles:", available_profiles())

# --- config -----------------------------------------------------------------
TRAJECTORIES = "../output/trajectories"
EVALUATIONS = "../output/evaluations"
OUT = "../output/curated"

# Pull from one or more runs. A list concatenates runs (deduped by
# trajectory_id) so you can top up a run that had failures with another.
# RUNS = ["1782625975", "1782650354"]   # or a single id, or None for the latest
# RUNS = ["1782539576", "1782585279"]   # or a single id, or None for the latest
RUNS = ["1784057990"]

PROFILE = "generous"        # see available_profiles() for the shipped set
HOLDOUT_FRACTION = 0.0      # >0 reserves a deterministic persona slice as "holdout"
MIN_TURNS = None            # set an int to require multi-turn conversations

# Assistant language-match gate: fraction of ASSISTANT turns whose language
# lingua detects as the requested one. Adjust or disable the profile's floor.
#   0.5  = profile default for `generous` (recommended for clean training data:
#          drop a trajectory if the assistant answered in-language <50% of turns)
#   0.3  = lenient: keep partial-language trajectories (low-resource languages
#          where the model code-switches to English for technical terms)
#   None = drop this gate entirely: rely on in-sim native-script enforcement +
#          manual review. Useful for the India multi-language PREVIEW, where the
#          assistant's language adherence is itself the finding you want to keep,
#          not a filter (otherwise Tamil/Odia/etc. get thinned out).
# Only bites lingua-detectable locales; romanized (_Latn_IN) and lingua-less
# native locales (Odia/Kannada/Malayalam/Nepali/Marathi) don't carry this axis.
REQUESTED_LANGUAGE_MATCH_RATE = 0
MAX_RECORDS = None          # global cap to N highest-quality records (None = keep all)
MAX_RECORDS_PER_LOCALE = 10 # cap to N highest-quality passing records PER LOCALE
                            # (applied first; composes with MAX_RECORDS). None = no per-locale cap.
STRATIFY_BY = ["probe_family"]   # balance the GLOBAL cap across these cols; None = global top-K

# Exact columns written to parquet / uploaded to HF. Edit this list to write
# fewer or more: any name absent from the data is silently skipped, and
# locale/probe_family are always kept (the partitioned writer needs them).
# Run `result.curated.columns` (after the select cell) to see every column
# available to add (e.g. selection_quality_score, simulation_outcome, ...).
# Entries are resolved through `selection.resolve_columns`, so you can also
# drop in LOGICAL GROUP TOKENS that expand to several columns and are skipped
# where absent: e.g. "persona", "conversation", or "finance" (the
# financial_services gold-metadata flywheel signal: tier, gold_tool_sequence,
# expected_state_deltas, retrieved_document_ids, ...). Named schemas
# ("compact"/"full") are available the same way.
COLUMNS_TO_WRITE = [
    "run",
    "locale",
    "probe_family",
    "probe_type",
    "probe_variant",
    "theme",
    "num_turns",
    "num_tools",
    "tools",
    "trajectory_id",
    "persona_uuid",
    "persona_name",
    "persona_age",
    "persona_sex",
    "persona_education_level",
    "persona_occupation",
    "persona_location",
    # "persona_country",
    # "behavioral_profile",
    "disclosure_style",
    "user_interaction_style",
    # "conversation_language",
    "conversation_messages",
    # "conversation_metadata",
    # "conversation_status",
    # "finance",   # logical group: financial_services gold metadata (task_tier,
    #              # gold_tool_sequence, expected_state_deltas, retrieved_document_ids,
    #              # institution_type, dynamic_category_id, ...). Expands when present,
    #              # silently skipped for non-finance rows. Uncomment for finance runs.
]

# Generation LLM (assistant under test) - recorded in the HF dataset card /
# manifest for provenance. Not stored in the parquet, so set it explicitly.
GENERATION_MODEL = "openai/openai/gpt-5.5"#"google/gemma-4-31B-it"

profile = get_profile(PROFILE)
overrides = {}
if MIN_TURNS is not None:
    overrides["min_turns"] = int(MIN_TURNS)
if HOLDOUT_FRACTION:
    overrides["holdout_fraction"] = float(HOLDOUT_FRACTION)
if overrides:
    profile = dataclasses.replace(profile, **overrides)

# Adjust (or drop) the assistant language-match floor per the knob above.
# axis_floors is a frozen mapping, so build a new dict and replace.
_LANG_AXIS = "language.requested_language_match_rate"
if REQUESTED_LANGUAGE_MATCH_RATE is None:
    _floors = {k: v for k, v in profile.axis_floors.items() if k != _LANG_AXIS}
else:
    _floors = {**profile.axis_floors, _LANG_AXIS: float(REQUESTED_LANGUAGE_MATCH_RATE)}
if _floors != dict(profile.axis_floors):
    profile = dataclasses.replace(profile, axis_floors=_floors)
    print(
        f"language-match floor: "
        + ("dropped" if REQUESTED_LANGUAGE_MATCH_RATE is None
           else f"{REQUESTED_LANGUAGE_MATCH_RATE}")
        + f" (profile default was {get_profile(PROFILE).axis_floors.get(_LANG_AXIS)})"
    )

# --- load + select (no write yet) -------------------------------------------
traj_df, eval_df, run_id = load_runs(TRAJECTORIES, EVALUATIONS, runs=RUNS)
result = select_trajectories(
    traj_df, eval_df, profile=profile,
    max_records=MAX_RECORDS, max_per_locale=MAX_RECORDS_PER_LOCALE,
    stratify_by=STRATIFY_BY,
)

s = result.summary
_caps = []
if s.max_per_locale is not None:
    _caps.append(f"<={s.max_per_locale}/locale")
if s.max_records is not None:
    _caps.append(f"<={s.max_records} total")
print(f"runs {run_id} · profile {s.profile}")
print(f"  passed {s.passed}/{s.total}  (dropped {s.dropped}) · selected {s.selected}"
      + (f"  (caps: {', '.join(_caps)})" if _caps else ""))
for reason, count in sorted(s.drop_reasons.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"    drop: {reason:40s} {count}")
if s.holdout:
    print(f"  holdout split: {s.holdout}")

In [ ]:
# Preview EXACTLY what will be written/uploaded (the COLUMNS_TO_WRITE columns),
# before any parquet/HF write happens. Edit COLUMNS_TO_WRITE above and re-run.
import json

import pandas as pd

from selection import project

try:
    from IPython.display import display
except ImportError:  # running headless / as a script
    display = print

payload = project(result.curated, COLUMNS_TO_WRITE)
cols = list(payload.columns)
print(f"{len(cols)} columns, {len(payload)} rows")
print("columns:", cols)
dropped = [c for c in COLUMNS_TO_WRITE if c not in cols]
if dropped:
    print("requested but absent (skipped):", dropped)

# Where the kept records come from (per source run, per locale/probe_family).
if "run" in payload.columns:
    print("\nselected per source run:")
    for run, count in payload["run"].astype(str).value_counts().sort_index().items():
        print(f"  run {run:14s} {count}")
print("\nselected by locale/probe_family:")
for group, count in sorted(result.summary.kept_by_group.items()):
    print(f"  {group:40s} {count}")

# First rows of the projected payload (ALL columns that will be written;
# long cell values are truncated for readability, not dropped).
with pd.option_context(
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", 40,
):
    display(payload.head(5))

# A sample conversation from the highest-quality kept record.
if len(result.curated):
    row = result.curated.sort_values("selection_quality_score", ascending=False).iloc[0]
    print(
        f"\ntop-quality kept trajectory {row['trajectory_id']} "
        f"(quality={row['selection_quality_score']:.3f}, "
        f"axes={row['selection_axis_scores']}):"
    )
    for msg in json.loads(row["conversation_messages"]):
        text = msg["content"].replace("\n", " ")
        print(f"  [{msg['role']:9s}] {text[:160]}")

In [ ]:
# Write the curated subset to parquet (projected to COLUMNS_TO_WRITE),
# namespaced by run set + profile. Run once the preview above looks right.
from selection import write_curated_dataset

out_root = write_curated_dataset(
    result.curated, OUT, run_id, profile,
    summary=result.summary, columns=COLUMNS_TO_WRITE,
)
print(f"wrote {result.summary.selected} records ({len(payload.columns)} cols) -> {out_root}")

In [ ]:
# Optional Hugging Face upload. Leave HF_REPO = None to skip.
# Auth via HF_TOKEN env var or a cached `huggingface-cli login`.
# HF_REPO = "nvidia/usersim-rlhf-general-hi-latn-in-1k"             # e.g. "nvidia/usersim-rlhf-general-hi-deva-in-1k"
HF_REPO = "nvidia/usersim-rlhf-general-all-indic-languages-120-gpt-5.5"
HF_PRIVATE = True


# Hugging Face credentials (only prompts when HF_REPO is set). The token is a
# write-scoped HF access token (https://huggingface.co/settings/tokens). It is
# read here via getpass so it never gets baked into the notebook; falls back to
# the HF_TOKEN env var / a cached `huggingface-cli login` if you leave it blank.
import getpass
import os

HF_TOKEN = None
if HF_REPO:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
        "Hugging Face write token (blank = use cached login): "
    ) or None
    print("HF token captured." if HF_TOKEN else "No token entered: using cached HF login.")
else:
    print("HF_REPO not set: skipping credential prompt.")

In [ ]:
# Optional: upload the curated subset to a (private) Hugging Face dataset repo.
# No-op unless HF_REPO is set above. Uploads the COLUMNS_TO_WRITE columns as
# a single parquet split + a dataset card + the selection manifest, using the
# token captured by the getpass cell above.
from selection import build_manifest, push_to_hf
from selection.io import _provenance

if HF_REPO:
    manifest = build_manifest(
        profile=profile, summary=result.summary, source_run=str(run_id),
        eval_column="assistant_eval",
        provenance=_provenance(result.curated, "assistant_eval"),
        generation_model=GENERATION_MODEL,
    )
    url = push_to_hf(
        result.curated, HF_REPO, private=HF_PRIVATE, token=HF_TOKEN,
        manifest=manifest, profile=profile, columns=COLUMNS_TO_WRITE,
    )
    print(f"uploaded {len(result.curated)} rows -> {url} (private={HF_PRIVATE})")
else:
    print("HF_REPO not set: skipping upload.")